In [8]:
import pandas as pd
import re
import json

In [9]:
import pandas as pd
import pymysql
import re

# ======================================================
# 1. 영업시간 파싱 함수
# ======================================================

DAY_MAP = {"월":0,"화":1,"수":2,"목":3,"금":4,"토":5,"일":6,"매일":"ALL"}

def parse_oper_time(text):
    """
    '월~금 09:00~18:00 토 09:00~15:00'
    → [(day_of_week, open, close, is_next_day), ...]
    """
    if text is None or str(text).strip() == "":
        return []

    result = []

    tokens = re.findall(
        r'(매일\b|[월화수목금토일](?:~[월화수목금토일])?)\s*([\d]{2}:[\d]{2}~[\d]{2}:[\d]{2})',
        str(text)
    )


    for day_part, time_part in tokens:
        # 요일 파싱
        if day_part == "매일":
            days = list(range(7))  # 0~6
        elif "~" in day_part:
            s, e = day_part.split("~")
            days = list(range(DAY_MAP[s], DAY_MAP[e] + 1))
        else:
            days = [DAY_MAP[day_part]]

        # 시간 파싱
        open_t, close_t = time_part.split("~")
        is_next_day = int(close_t < open_t)

        for d in days:
            result.append((d, open_t, close_t, is_next_day))

    return result


# ======================================================
# 2. 엑셀 / CSV 로드
# ======================================================

df = pd.read_csv("/KC_PET_ACP_CTLSTT_LC_DATA_2023.csv")

# 컬럼명 안전 처리
df.columns = df.columns.str.strip()

print(df.columns)


# ======================================================
# 3. DB 연결
# ======================================================

conn = pymysql.connect(
    host="localhost",
    user="mini",
    password="mini",
    db="miniproject",
    charset="utf8mb4"
)
cursor = conn.cursor()


# ======================================================
# 4. place 테이블 → {fclty_nm: id} 매핑
# ======================================================

cursor.execute("SELECT id, fclty_nm, LNM_ADDR FROM place")
place_map = {
    (str(name).strip(), str(addr).strip()): pid
    for pid, name, addr in cursor.fetchall()
}


Index(['FCLTY_NM', 'CTGRY_ONE_NM', 'CTGRY_TWO_NM', 'CTGRY_THREE_NM',
       'CTPRVN_NM', 'SIGNGU_NM', 'LEGALDONG_NM', 'LI_NM', 'LNBR_NO', 'ROAD_NM',
       'BULD_NO', 'LC_LA', 'LC_LO', 'ZIP_NO', 'RDNMADR_NM', 'LNM_ADDR',
       'TEL_NO', 'HMPG_URL', 'RSTDE_GUID_CN', 'OPER_TIME', 'PARKNG_POSBL_AT',
       'UTILIIZA_PRC_CN', 'PET_POSBL_AT', 'PET_INFO_CN',
       'ENTRN_POSBL_PET_SIZE_VALUE', 'PET_LMTT_MTR_CN',
       'IN_PLACE_ACP_POSBL_AT', 'OUT_PLACE_ACP_POSBL_AT', 'FCLTY_INFO_DC',
       'PET_ACP_ADIT_CHRGE_VALUE', 'LAST_UPDT_DE'],
      dtype='object')


In [10]:
# ======================================================
# 5. INSERT SQL
# ======================================================

sql = """
INSERT INTO operationdata (
    place_id,
    day_of_week,
    open_time,
    close_time,
    is_next_day
)
VALUES (%s, %s, %s, %s, %s)
"""


# ======================================================
# 6. 엑셀 → 파싱 → INSERT
# ======================================================

for _, r in df.iterrows():
    place_name = str(r["FCLTY_NM"]).strip()
    place_addr = str(r["LNM_ADDR"]).strip()  # 주소 컬럼
    oper_text = r["OPER_TIME"]

    place_id = place_map.get((place_name, place_addr))
    if place_id is None:
        print(f"❗ place_id 매핑 실패: {place_name} / {place_addr}")
        continue

    rows = parse_oper_time(oper_text)

    for day_of_week, open_t, close_t, is_next_day in rows:
        cursor.execute(
            sql,
            (place_id, day_of_week, open_t, close_t, is_next_day)
        )


# ======================================================
# 7. 커밋 & 종료
# ======================================================

conn.commit()
conn.close()

print("✅ operationdata INSERT 완료")


✅ operationdata INSERT 완료
